## Imports

In [2]:
import numpy as np
import cv2 as cv2
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from ultralytics import YOLO

## Load Pretrained Model

In [3]:
model = MobileNetV2(weights='imagenet')
detector = YOLO('yolov8n.pt')

## Define Waste Category Mapping

In [4]:
WASTE_MAP = {
    "banana": "Compost",
    "banana peel": "Compost",
    "apple": "Compost",
    "orange": "Compost",
    "bottle": "Recycle",
    "water bottle": "Recycle",
    "plastic bottle": "Recycle",
    "can": "Recycle",
    "carton": "Recycle",
    "paper towel": "Compost",
    "paper": "Recycle",
    "cup": "Landfill",
    "fork": "Landfill",
    "spoon": "Landfill",
    "knife": "Landfill",
    "biscuit": "Compost",
}

def classify_waste(label):
    label = label.lower()
    for key in WASTE_MAP:
        if key in label:
            return WASTE_MAP[key]
    return "Landfill" 

## Webcam + Frame Classify

In [8]:
## this script is object detection

cap = cv2.VideoCapture(0)
### press q to quit
print("Starting TrashCam... press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = detector(frame, verbose=False)

    if len(results[0].boxes) == 0:
        cv2.putText(frame, "No object detected", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow("TrashCam", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    areas = [(x2-x1)*(y2-y1) for (x1,y1,x2,y2) in boxes]
    idx = areas.index(max(areas))
    x1, y1, x2, y2 = boxes[idx].astype(int)

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    crop = frame[y1:y2, x1:x2]

    img = cv2.resize(crop, (224, 224))
    img = preprocess_input(np.expand_dims(img.astype("float32"), axis=0))

    preds = model.predict(img, verbose=0)
    decoded = decode_predictions(preds, top=1)[0][0]
    label = decoded[1]
    confidence = decoded[2]

    waste_type = classify_waste(label)

    text = f"{label} ({confidence:.2f}) = {waste_type}"
    cv2.putText(frame, text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("TrashCam", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Starting TrashCam... press 'q' to quit.


In [ ]:
cap = cv2.VideoCapture(0)

print("Starting TrashCam... press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # --- BOX OVERLAY LOGIC ---
    h, w, _ = frame.shape
    box_w, box_h = 300, 300
    x1 = w//2 - box_w//2
    y1 = h//2 - box_h//2
    x2 = x1 + box_w
    y2 = y1 + box_h

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    crop = frame[y1:y2, x1:x2]

    # --- HAND REJECTION (skin detection) ---
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    # Skin color range (works well for most tones)
    lower_skin = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin = np.array([20, 255, 255], dtype=np.uint8)

    mask = cv2.inRange(hsv, lower_skin, upper_skin)
    skin_pixels = cv2.countNonZero(mask)

    # If too many skin pixels → skip classification
    if skin_pixels > 5000:
        cv2.putText(frame, "Hand detected - waiting...", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow("TrashCam", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # --- NORMAL CLASSIFICATION ---
    img = cv2.resize(crop, (224, 224))
    img = preprocess_input(np.expand_dims(img.astype("float32"), axis=0))

    preds = model.predict(img, verbose=0)
    decoded = decode_predictions(preds, top=1)[0][0]
    label = decoded[1]
    confidence = decoded[2]

    waste_type = classify_waste(label)

    text = f"{label} ({confidence:.2f}) = {waste_type}"
    cv2.putText(frame, text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("TrashCam", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Starting TrashCam... press 'q' to quit.
